# Task 3: RAG Core Logic & Evaluation

**Objective:** Build the retrieval and generation pipeline using the pre-built vector store and evaluate its effectiveness.

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import os, json, time
import faiss
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 1. Load Pre-built Vector Store from Parquet

In [ ]:
paths = [
    '../data/raw/complaint_embeddings.parquet',
    r'c:\KAIM\rag-complaint-chatbot\data\raw\complaint_embeddings.parquet',
]
for p in paths:
    if os.path.exists(p):
        df_emb = pd.read_parquet(p)
        print(f'Loaded parquet: {df_emb.shape}')
        print(f'Columns: {df_emb.columns.tolist()}')
        break

df_emb.head(3)

In [ ]:
# Identify embedding column and text column
print('Column dtypes:')
print(df_emb.dtypes)
print('\nSample values:')
for col in df_emb.columns:
    val = df_emb[col].iloc[0]
    print(f'{col}: {type(val).__name__} -> {str(val)[:80]}')

In [ ]:
# Find embedding column (array/list type)
emb_col = None
text_col = None
for col in df_emb.columns:
    val = df_emb[col].iloc[0]
    if isinstance(val, (list, np.ndarray)) and len(val) > 10:
        emb_col = col
        print(f'Embedding column: {col} (dim={len(val)})')
    if isinstance(val, str) and len(val) > 20:
        if text_col is None:
            text_col = col
            print(f'Text column: {col}')

# Find metadata columns
product_col = next((c for c in df_emb.columns if 'product' in c.lower()), None)
print(f'Product column: {product_col}')

In [ ]:
# Build FAISS index from parquet embeddings
print('Building FAISS index from pre-built embeddings...')

embeddings = np.array(df_emb[emb_col].tolist()).astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

# Normalize for cosine similarity
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)
embeddings_norm = embeddings / norms

dim = embeddings_norm.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings_norm)

print(f'FAISS index built! Total vectors: {index.ntotal:,}')

# Extract chunks and metadata
chunks = df_emb[text_col].tolist()
metadata = []
for _, row in df_emb.iterrows():
    meta = {}
    for col in df_emb.columns:
        if col != emb_col:
            meta[col] = str(row[col])
    metadata.append(meta)

print(f'Chunks: {len(chunks):,}')
print(f'Sample chunk: {chunks[0][:200]}')

## 2. Load Embedding Model

In [ ]:
print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded! Dim: {embed_model.get_sentence_embedding_dimension()}')

## 3. Retriever Function

In [ ]:
def retrieve(query: str, k: int = 5, product_filter: str = None) -> list:
    """
    Retrieve top-k most relevant complaint chunks for a query.
    
    Args:
        query: User question in plain English
        k: Number of chunks to retrieve
        product_filter: Optional product category filter
    
    Returns:
        List of dicts with text, score, and metadata
    """
    # Embed the query
    q_emb = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    q_emb = q_emb / np.linalg.norm(q_emb)
    
    # Search with extra buffer for filtering
    search_k = k * 5 if product_filter else k
    scores, indices = index.search(q_emb, search_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(chunks):
            continue
        meta = metadata[idx]
        # Apply product filter if specified
        if product_filter:
            prod = meta.get(product_col, '').lower()
            if product_filter.lower() not in prod:
                continue
        results.append({
            'text': chunks[idx],
            'score': float(score),
            'metadata': meta
        })
        if len(results) >= k:
            break
    return results

# Quick test
test = retrieve('credit card billing dispute', k=2)
for r in test:
    print(f'Score: {r["score"]:.3f}')
    print(f'Text: {r["text"][:150]}...')
    print()

## 4. Prompt Template & Generator

In [ ]:
PROMPT_TEMPLATE = """You are a financial analyst assistant for CrediTrust Financial.
Your task is to answer questions about customer complaints based ONLY on the retrieved complaint excerpts below.
If the context does not contain enough information to answer, state clearly that you don't have enough information.
Always cite specific examples from the complaints when possible.
Be concise and actionable — your audience is a Product Manager who needs to act on insights.

Retrieved Complaint Excerpts:
{context}

Question: {question}

Answer based on the complaint excerpts above:"""

def build_context(results: list) -> str:
    """Format retrieved chunks into a context string."""
    context_parts = []
    for i, r in enumerate(results):
        meta = r['metadata']
        prod = meta.get(product_col, 'Unknown')
        context_parts.append(
            f"[Excerpt {i+1} | Product: {prod} | Relevance: {r['score']:.2f}]\n{r['text']}"
        )
    return '\n\n'.join(context_parts)

print('Prompt template ready!')

In [ ]:
# Use a lightweight LLM via HuggingFace pipeline
# We use a small text-generation model that works on CPU
from transformers import pipeline

print('Loading LLM...')
try:
    generator = pipeline(
        'text-generation',
        model='facebook/opt-125m',
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    USE_LLM = True
    print('LLM loaded!')
except Exception as e:
    print(f'LLM load failed: {e}')
    print('Will use extractive summarization as fallback')
    USE_LLM = False

In [ ]:
def generate_answer(question: str, context: str) -> str:
    """Generate answer from context using LLM or fallback."""
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    
    if USE_LLM:
        try:
            output = generator(prompt, max_new_tokens=300)[0]['generated_text']
            # Extract only the answer part
            answer_start = output.find('Answer based on')
            if answer_start != -1:
                answer = output[answer_start:].split('\n', 2)[-1].strip()
            else:
                answer = output[len(prompt):].strip()
            return answer if answer else output[-300:]
        except Exception as e:
            return f'Generation error: {e}'
    else:
        # Extractive fallback: return most relevant sentences
        lines = context.split('. ')
        q_words = set(question.lower().split())
        scored = [(sum(w in line.lower() for w in q_words), line)
                  for line in lines if len(line) > 30]
        scored.sort(reverse=True)
        top = [s[1] for s in scored[:4]]
        return 'Based on retrieved complaints: ' + '. '.join(top)


def rag_query(question: str, k: int = 5, product_filter: str = None) -> dict:
    """Full RAG pipeline: retrieve -> build context -> generate answer."""
    results = retrieve(question, k=k, product_filter=product_filter)
    context = build_context(results)
    answer = generate_answer(question, context)
    return {
        'question': question,
        'answer': answer,
        'sources': results,
        'context': context
    }

print('RAG pipeline ready!')

## 5. Qualitative Evaluation — 8 Test Questions

In [ ]:
test_questions = [
    ('Why are customers unhappy with credit cards?', None),
    ('What are the most common issues with money transfers?', 'Money Transfer'),
    ('What problems do customers face with personal loans?', 'Personal Loan'),
    ('Why are savings account holders complaining about closing accounts?', 'Savings Account'),
    ('How are customers affected by unauthorized transactions?', None),
    ('What billing disputes are customers reporting on credit cards?', 'Credit Card'),
    ('Are there complaints about fraud in money transfers?', 'Money Transfer'),
    ('What issues do customers face when trying to get a loan?', 'Personal Loan'),
]

print(f'Running {len(test_questions)} evaluation queries...\n')
eval_results = []

for q, product in test_questions:
    print(f'Q: {q}')
    result = rag_query(q, k=5, product_filter=product)
    eval_results.append(result)
    print(f'A: {result["answer"][:300]}')
    print(f'Sources: {len(result["sources"])} chunks retrieved')
    print('-' * 60)

In [ ]:
# Print evaluation table
print('\n=== EVALUATION TABLE ===')
print(f'{"Question":<50} {"Sources":<8} {"Score":<6} Comments')
print('-' * 100)
for i, (r, (q, prod)) in enumerate(zip(eval_results, test_questions)):
    n_sources = len(r['sources'])
    avg_score = np.mean([s['score'] for s in r['sources']]) if r['sources'] else 0
    print(f'{q[:48]:<50} {n_sources:<8} {avg_score:.3f}')

In [ ]:
# Save evaluation results
eval_export = []
for r, (q, prod) in zip(eval_results, test_questions):
    eval_export.append({
        'question': q,
        'product_filter': prod,
        'answer': r['answer'][:500],
        'top_source_1': r['sources'][0]['text'][:200] if r['sources'] else '',
        'top_source_2': r['sources'][1]['text'][:200] if len(r['sources']) > 1 else '',
        'avg_retrieval_score': np.mean([s['score'] for s in r['sources']]) if r['sources'] else 0
    })

os.makedirs('../data/processed', exist_ok=True)
pd.DataFrame(eval_export).to_csv('../data/processed/rag_evaluation.csv', index=False)
print('Evaluation saved to data/processed/rag_evaluation.csv')

## 6. Save RAG Pipeline as Module
The RAG logic is also implemented in `src/rag_pipeline.py` for use by the Gradio app.

In [ ]:
rag_module = '''
# src/rag_pipeline.py
import numpy as np
import pandas as pd
import faiss
import os
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline

PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust Financial.
Answer questions about customer complaints using ONLY the retrieved excerpts.
If context is insufficient, say so. Be concise and actionable.

Retrieved Complaint Excerpts:
{context}

Question: {question}

Answer:"""

class RAGPipeline:
    def __init__(self, parquet_path: str, emb_col: str, text_col: str, product_col: str):
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        df = pd.read_parquet(parquet_path)
        self.chunks = df[text_col].tolist()
        self.metadata = df.drop(columns=[emb_col]).to_dict("records")
        self.product_col = product_col
        embs = np.array(df[emb_col].tolist()).astype("float32")
        embs = embs / np.linalg.norm(embs, axis=1, keepdims=True)
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)
        try:
            self.generator = hf_pipeline("text-generation", model="facebook/opt-125m",
                max_new_tokens=300, do_sample=False)
        except:
            self.generator = None

    def retrieve(self, query, k=5, product_filter=None):
        q = self.embed_model.encode([query], convert_to_numpy=True).astype("float32")
        q = q / np.linalg.norm(q)
        scores, indices = self.index.search(q, k * 5)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0: continue
            meta = self.metadata[idx]
            if product_filter and product_filter.lower() not in str(meta.get(self.product_col,"")).lower():
                continue
            results.append({"text": self.chunks[idx], "score": float(score), "metadata": meta})
            if len(results) >= k: break
        return results

    def query(self, question, k=5, product_filter=None):
        results = self.retrieve(question, k=k, product_filter=product_filter)
        context = "\\n\\n".join([f"[Excerpt {i+1}]\\n{r[\x27text\x27]}" for i, r in enumerate(results)])
        prompt = PROMPT_TEMPLATE.format(context=context, question=question)
        if self.generator:
            out = self.generator(prompt)[0]["generated_text"][len(prompt):].strip()
        else:
            out = "LLM unavailable. Top complaint: " + (results[0]["text"][:300] if results else "No results found.")
        return {"answer": out, "sources": results}
'''

os.makedirs('../src', exist_ok=True)
with open('../src/rag_pipeline.py', 'w') as f:
    f.write(rag_module)
print('src/rag_pipeline.py saved!')